In [95]:
import numpy as np
from setup_mods import read_param_file, initialize_data
from galario.double import get_image_size
import ast

In [92]:
pwd

'/arc/home/pknowlton/git_repo/ALMA-Galaxy-Visibility-Modelling-PFK/radial_version'

In [96]:
def initialize_data(param_file):

    params={}
    with open(param_file, 'r') as f:
        for line in f:
            line = line.split("#", 1)[0].strip() #This will skip in-line comments
            if "=" in line:
                pkey, pvalue = line.split("=", maxsplit=1)
                pkey = pkey.strip()
                pvalue = ast.literal_eval(pvalue.strip())
                if isinstance(pvalue, list):
                    pvalue = np.array(pvalue, dtype=float)
                params[pkey] = pvalue

    u, v, re, im, w = np.require(np.loadtxt(params['visFile'], unpack=True), requirements='C')
    wavelength = 299792458/params['obsFreq']
    u /= wavelength
    v /= wavelength

    nx, dx = get_image_size(u, v)

    return params['radiusStart'], params['radiusStep'], params['radiusNumSteps'], nx, dx, u, v, re, im, w

In [97]:
args = initialize_data('/arc/home/pknowlton/git_repo/ALMA-Galaxy-Visibility-Modelling-PFK/radial_version/global_fitting_params.txt')

In [98]:
args

(0,
 0.001,
 25000,
 4096,
 1.8983707832917397e-07,
 array([ 22350.41036289,  48724.0436182 , -23514.47206854, ...,
          1508.16601931, -22055.15430278, -23563.3184608 ]),
 array([ 5.67048418e+04,  2.59049300e+04,  5.05225438e+04, ...,
         1.85920518e+05, -2.27975382e+01, -1.85943319e+05]),
 array([-0.03102749,  0.01100424,  0.02566671, ..., -0.03183961,
         0.02330604, -0.01934953]),
 array([ 0.01602061, -0.03702213, -0.01300304, ..., -0.01824165,
        -0.05576196, -0.02900367]),
 array([ 913.0098,  941.8783,  947.345 , ..., 1114.625 , 1170.349 ,
        1174.875 ]))

In [86]:
params

{'visFile': '/arc/projects/uvdisk_fit/ngc_3351/JS_data/cont93GHz/M95_C5+C2_cont93_uvtable.txt',
 'obsFreq': 93000000000.0,
 'numWalkers': 100,
 'burnTime': 150,
 'runTime': 3000,
 'radiusStart': 0,
 'radiusStep': 0.001,
 'radiusNumSteps': 25000,
 'dataFITSFile': '/arc/projects/uvdisk_fit/ngc_3351/JS_data/cont93GHz/M95_C5+C2_cont93_pbcorr_round.fits',
 'plotCenterRA': '10h43m57.75s',
 'plotCenterDec': '11:42:13.34deg',
 'plotBoxSide': 40,
 'plotName': 'Perturbed Initial Guess',
 'uvbinSize': 0}

In [87]:
data = initialize_data(params['visFile'], params['obsFreq'])

In [88]:
data[0]

4096

In [6]:
def log_print(prin_pars):
    print(prin_pars)

In [7]:
def log_prior(pars):

    peak, sigma, ring_rad, inclination, posangle, dRA, dDec = pars

    print(peak, ring_rad)
    log_print(pars)

    for i in (pars):
        print(i)

In [8]:
pars = [8, 2, 4, 9, 1, 2, 9]

In [9]:
log_prior(pars)

8 4
[8, 2, 4, 9, 1, 2, 9]
8
2
4
9
1
2
9


In [6]:
model_fits_ranges = [[-5, 15], [0, 10], [0, 20], [0, 90], [0, 180],[-5, 5], [-5, 5]]

In [ ]:
model_fit_ranges

In [21]:
pwd

'/arc/home/pknowlton/git_repo/ALMA-Galaxy-Visibility-Modelling-PFK/radial_version'

In [77]:
def log_prior(pars):

    model_fits_ranges = [[-5, 15], [0, 10], [0, 20], [0, 90], [0, 180],[-5, 5], [-5, 5]]

    for i in range(len(pars)):
        if not model_fits_ranges[i][0] < pars[i] < model_fits_ranges[i][1]:
            return -np.inf
    return 0.0
            

In [78]:
pars = [8, 2, 4, 9, 1, 2, 4]

In [80]:
log_prior(pars)

0.0

In [81]:
def log_prob(pars):
    lp = log_prior(pars)
    if not np.isfinite(lp):
        return -np.inf
    else:
        return 0.0

In [82]:
log_prob(pars)

0.0

In [99]:
model_fits_ranges = [[-5, 15], [0, 10], [0, 20], [0, 90], [0, 180],[-5, 5], [-5, 5]]
model_fits_initial_guesses = [6, 1, 7, 60, 15, 0, 0]

In [101]:
nwalkers = 32
ndim = len(model_fits_ranges)
pos = np.zeros([nwalkers, ndim])

In [115]:
for i in range(nwalkers):
    c = 0
    while c != 1:
        pos[i, :] = model_fits_initial_guesses + 1e-5*np.random.randn(ndim)
        chk_pars = log_prior(pos[i,:])
        if chk_pars == 0:
            c = 1
print('All guesses within prior range')

All guesses within prior range


In [116]:
pos

array([[ 6.00001802e+00,  9.99999370e-01,  6.99999808e+00,
         6.00000054e+01,  1.49999915e+01,  8.41291060e-06,
        -1.18977208e-05],
       [ 6.00001955e+00,  1.00000930e+00,  6.99998976e+00,
         5.99999934e+01,  1.50000093e+01,  1.12688035e-05,
        -1.25377597e-05],
       [ 6.00000330e+00,  9.99985231e-01,  6.99999673e+00,
         5.99999974e+01,  1.50000001e+01,  4.51842150e-06,
        -4.23089650e-06],
       [ 5.99997769e+00,  1.00000420e+00,  6.99998557e+00,
         5.99999912e+01,  1.50000124e+01,  1.09830121e-05,
        -7.51695600e-06],
       [ 6.00002747e+00,  9.99993247e-01,  7.00001886e+00,
         5.99999997e+01,  1.50000080e+01, -1.62110706e-05,
         6.81754686e-06],
       [ 6.00000393e+00,  9.99998201e-01,  6.99998209e+00,
         6.00000114e+01,  1.49999845e+01, -3.92416151e-06,
        -2.27323188e-06],
       [ 6.00000487e+00,  9.99997414e-01,  7.00002304e+00,
         5.99999948e+01,  1.50000081e+01, -4.70909917e-06,
        -2.5291492